## NeuroMTA Framework Example
### Example 1: Simple Core Model

This example demonstrates a simple NPU core model built using the NeuroMTA simulator. The simple architecture consists of a single Matrix Multiplication Unit (MXU) coupled with L1 scratchpad memory bank. The primary goal is to illustrate how this basic core can be utilized to execute a matrix multiplication operation. Assume that the MXU is a simple output-stationary systolic array-like computation unit, composed of a 128x128 MAC (Multiply-Accumulate Unit) array and PE registers. The precision that this NPU supports is INT32 only.

In [1]:
import torch
from neuromta.framework import *

In [2]:
class SimpleNPUCore(Core):
    def __init__(self, core_id):
        super().__init__(core_id, SimpleNPUCoreCycleModel())
    
        self.l1_memory = MemoryHandle( 
            base_addr=0x00, 
            bank_size=parse_mem_cap_str("2MB"),
            n_banks=1,
        )
        
        self.mxu_pe_arr = torch.zeros((128, 128), dtype=torch.int32)
    
    @core_command_method
    def mxu_compute(
        self, 
        
        ifm: DataContainer[torch.Tensor], 
        wgt: DataContainer[torch.Tensor], 
        psum: DataContainer[torch.Tensor], 
        ofm: DataContainer[torch.Tensor],
    
        preload_psum: bool = True,
        flush_ofm: bool = True,
    ):
        if preload_psum:
            psum.data = psum.data.view(torch.int32).reshape(128, 128)
            self.mxu_pe_arr[:, :] = psum.data

        ifm.data = ifm.data.view(torch.int32).reshape(128, 128)
        wgt.data = wgt.data.view(torch.int32).reshape(128, 128)
        
        self.mxu_pe_arr[:, :] = torch.matmul(ifm.data, wgt.data) + self.mxu_pe_arr

        if flush_ofm:
            ofm.data = self.mxu_pe_arr.clone()
            self.mxu_pe_arr[:, :] = 0
        
    @core_command_method
    def l1_read_single_page(self, ptr: Pointer, container: DataContainer[torch.Tensor]):
        container.data = self.l1_memory.get_data(ptr, size=128*128*4, dtype=torch.int32).reshape(128, 128)

    @core_command_method
    def l1_write_single_page(self, ptr: Pointer, container: DataContainer[torch.Tensor]):
        size = container.data.numel() * container.data.element_size()
        data = container.data.view(torch.int32).reshape(-1)
        self.l1_memory.set_data(ptr, size, data)

class SimpleNPUCoreCycleModel(CoreCycleModel):
    def __init__(self):
        super().__init__()
        
    def mxu_compute(
        self,
        ifm: DataContainer[torch.Tensor],
        wgt: DataContainer[torch.Tensor],
        psum: DataContainer[torch.Tensor],
        ofm: DataContainer[torch.Tensor],
        preload_psum: bool = True,
        flush_ofm: bool = True,
    ):
        i = 1
        if preload_psum: i += 1
        if flush_ofm: i += 1
        return 128 * i 
    
    def l1_read_single_page(self, ptr: Pointer, container: DataContainer[torch.Tensor]):
        return 64
    
    def l1_write_single_page(self, ptr: Pointer, container: DataContainer[torch.Tensor]):
        return 64

In [3]:
class SimpleNPUDevice(Device):
    def __init__(self):
        super().__init__()
        
        self.npu_core = SimpleNPUCore(core_id=0)

In [4]:
@jit_prototype
def example_kernel(
    core: SimpleNPUCore, 
    
    ifm: Pointer,
    wgt: Pointer,
    psum: Pointer,
    ofm: Pointer,
):
    containers = [DataContainer() for _ in range(4)]
    
    core.l1_read_single_page(ifm, containers[0])
    core.l1_read_single_page(wgt, containers[1]) 
    core.l1_read_single_page(psum, containers[2]) 

    core.mxu_compute(*containers)

    core.l1_write_single_page(ofm, containers[3])

In [5]:
device = SimpleNPUDevice()
device.initialize()

In [6]:
logger.set_print_options(log_level=LogLevel.DEBUG)
device.set_command_debug_verbosity(verbose=True)

In [7]:
core = device.npu_core

ifm  = Pointer(addr=core.l1_memory.base_addr)
wgt  = Pointer(addr=core.l1_memory.base_addr + 128*128*4)
psum = Pointer(addr=core.l1_memory.base_addr + 2*128*128*4)
ofm  = Pointer(addr=core.l1_memory.base_addr + 3*128*128*4)

In [8]:
ifm_tensor = torch.randint(0, 10, (128, 128), dtype=torch.int32)
wgt_tensor = torch.randint(0, 10, (128, 128), dtype=torch.int32)
psum_tensor = torch.randint(0, 10, (128, 128), dtype=torch.int32)

core.l1_memory.set_data(ifm, 128*128*4, ifm_tensor)
core.l1_memory.set_data(wgt, 128*128*4, wgt_tensor)
core.l1_memory.set_data(psum, 128*128*4, psum_tensor)

In [9]:
kernel = example_kernel(core, ifm, wgt, psum, ofm)
kernel.dispatch(slot_id="main")

device.run_kernels()

[DEBUG] 0      - 64     | 0          | MAIN<main>::example_kernel                                                                           | command: l1_read_single_page
[DEBUG] 64     - 128    | 0          | MAIN<main>::example_kernel                                                                           | command: l1_read_single_page
[DEBUG] 128    - 192    | 0          | MAIN<main>::example_kernel                                                                           | command: l1_read_single_page
[DEBUG] 192    - 576    | 0          | MAIN<main>::example_kernel                                                                           | command: mxu_compute
[DEBUG] 576    - 640    | 0          | MAIN<main>::example_kernel                                                                           | command: l1_write_single_page


In [11]:
reference = torch.matmul(ifm_tensor, wgt_tensor) + psum_tensor
simulated = core.l1_memory.get_data(ofm, size=128*128*4, dtype=torch.int32).reshape(128, 128)

In [12]:
print(f"simulation terminated in {core.timestamp} cycles")
print("\nreference output:\n", reference)
print("\nsimulated output:\n", simulated)
print(f"\nsimulation {'PASSED' if torch.equal(reference, simulated) else 'FAILED'}")

simulation terminated in 640 cycles

reference output:
 tensor([[2924, 2467, 3019,  ..., 2602, 2463, 2938],
        [2593, 2303, 2725,  ..., 2226, 2227, 2464],
        [2275, 2143, 2643,  ..., 2045, 2091, 2170],
        ...,
        [2733, 2564, 2864,  ..., 2313, 2467, 2673],
        [2367, 2113, 2551,  ..., 2189, 2212, 2375],
        [2663, 2230, 2776,  ..., 2413, 2339, 2475]], dtype=torch.int32)

simulated output:
 tensor([[2924, 2467, 3019,  ..., 2602, 2463, 2938],
        [2593, 2303, 2725,  ..., 2226, 2227, 2464],
        [2275, 2143, 2643,  ..., 2045, 2091, 2170],
        ...,
        [2733, 2564, 2864,  ..., 2313, 2467, 2673],
        [2367, 2113, 2551,  ..., 2189, 2212, 2375],
        [2663, 2230, 2776,  ..., 2413, 2339, 2475]], dtype=torch.int32)

simulation PASSED
